# Mock Unity Socket Client for DLCLive Processor

This notebook acts as the **Unity-side socket client** for the legacy DLC socket processor.

## Which processor style this targets

The legacy processor chain you showed is:

```text
dlc_inference_w_pd_sync
    -> dlc_inference_w_pd
        -> MyProcessor_socket
```

`MyProcessor_socket` opens a `multiprocessing.connection.Listener`, defaulting to:

```python
("127.0.0.1", 6000)
authkey=b"secret password"
```

It sends payloads from `process()` shaped like:

```python
[time.time(), x, y, heading, head_angle, signal]
```

This notebook connects as the **client** and tries to catch those pose/kinematics packets.

## Important ordering note

The legacy `MyProcessor_socket` does **not** have a background accept thread. It accepts a client only when `process()` runs. That means if DLC inference is not producing poses yet, `Client(...)` may block or time out. If that happens, start/continue DLC inference and retry.

In [1]:
from __future__ import annotations

import json
import queue
import threading
import time
from dataclasses import asdict, dataclass
from multiprocessing.connection import Client
from pathlib import Path
from typing import Any

## Configuration

Adjust these if the processor uses a different port or auth key.

For the legacy processor, the defaults are usually:

```python
ADDRESS = ("127.0.0.1", 6000)
AUTHKEY = b"secret password"
```

In [2]:
ADDRESS = ("127.0.0.1", 6000)
AUTHKEY = b"secret password"

# How long to wait for a connection attempt before considering it failed.
CONNECT_TIMEOUT_S = 10.0

# How long the receive loop should poll while waiting for new packets.
POLL_INTERVAL_S = 0.05

print("Target processor socket:", ADDRESS)

Target processor socket: ('127.0.0.1', 6000)


## Client helpers

`multiprocessing.connection.Client(...)` can block if the server has not called `accept()` yet. To avoid freezing the notebook, `connect_with_timeout()` performs the connection attempt in a background thread.

In [3]:
class ConnectTimeoutError(TimeoutError):
    pass


def connect_with_timeout(address, authkey: bytes, timeout_s: float = 10.0):
    """Connect to a multiprocessing.connection.Listener without freezing the notebook forever."""
    result_q: queue.Queue[tuple[str, Any]] = queue.Queue(maxsize=1)

    def worker():
        try:
            conn = Client(address, authkey=authkey)
            result_q.put(("ok", conn))
        except Exception as exc:
            result_q.put(("error", exc))

    t = threading.Thread(target=worker, name="MockUnityConnect", daemon=True)
    t.start()
    t.join(timeout_s)

    if t.is_alive():
        raise ConnectTimeoutError(
            f"Timed out after {timeout_s:.1f}s while connecting to {address}. "
            "For legacy MyProcessor_socket, this can happen if DLC process() has not called listener.accept() yet."
        )

    status, payload = result_q.get_nowait()
    if status == "ok":
        return payload
    raise payload


@dataclass
class LegacyPosePacket:
    timestamp: float
    x: float
    y: float
    heading: float
    head_angle: float
    signal: float
    raw: Any


def decode_payload(payload: Any) -> dict[str, Any]:
    """Decode either legacy list payloads or newer dict/list mock payloads."""
    # Legacy MyProcessor_socket payload:
    # [time.time(), x, y, heading, head_angle, signal]
    if isinstance(payload, list) and len(payload) == 6:
        pkt = LegacyPosePacket(
            timestamp=float(payload[0]),
            x=float(payload[1]),
            y=float(payload[2]),
            heading=float(payload[3]),
            head_angle=float(payload[4]),
            signal=float(payload[5]),
            raw=payload,
        )
        return {"kind": "legacy_pose", **asdict(pkt)}

    # Newer/base mock payloads may be dictionaries.
    if isinstance(payload, dict):
        kind = payload.get("type", "dict")
        return {"kind": kind, "raw": payload}

    # Some processors broadcast [timestamp, pose].
    if isinstance(payload, list) and len(payload) == 2:
        return {"kind": "timestamp_pose", "timestamp": payload[0], "pose": payload[1], "raw": payload}

    return {"kind": "unknown", "raw": payload}

## Connect to the DLC processor socket

Run this cell once the processor has been created and its listener should be available.

If it times out, it likely means either:

1. the processor has not been instantiated yet,
2. the address/authkey are wrong,
3. the legacy processor is waiting until `process()` runs before accepting the connection,
4. another process is using the port.

In [4]:
conn = connect_with_timeout(ADDRESS, AUTHKEY, timeout_s=CONNECT_TIMEOUT_S)
print("Connected to processor socket:", ADDRESS)

Connected to processor socket: ('127.0.0.1', 6000)


## Optional: send a ping/status command

Only use this for processors that implement command handling, such as the newer `BaseProcessorSocket` or the standalone mock processor.

The legacy `MyProcessor_socket` does **not** read commands from the client, so skip this cell for that processor.

In [5]:
# Uncomment only for BaseProcessorSocket-style processors or the standalone mock.
# conn.send({"cmd": "ping"})
# if conn.poll(2.0):
#     print("Response:", conn.recv())
# else:
#     print("No response. This is expected for legacy MyProcessor_socket.")

## Receive pose packets

This cell listens for up to `duration_s` seconds and prints decoded packets. For legacy `MyProcessor_socket`, you should see `legacy_pose` packets once `process()` is called by DLC inference.

In [6]:
def receive_packets(conn, duration_s: float = 30.0, max_packets: int | None = None):
    start = time.time()
    packets: list[dict[str, Any]] = []

    while time.time() - start < duration_s:
        if conn.poll(POLL_INTERVAL_S):
            payload = conn.recv()
            decoded = decode_payload(payload)
            decoded["received_at"] = time.time()
            packets.append(decoded)
            print(json.dumps(decoded, default=str, indent=2))

            if max_packets is not None and len(packets) >= max_packets:
                break

    print(f"Received {len(packets)} packet(s).")
    return packets


packets = receive_packets(conn, duration_s=30.0, max_packets=20)

{
  "kind": "pose",
  "raw": {
    "type": "pose",
    "timestamp": 1783588105.750286,
    "step": 12,
    "pose": [
      [
        288.7191162109375,
        298.63250732421875,
        1.0
      ],
      [
        293.0001220703125,
        307.95672607421875,
        0.973543643951416
      ],
      [
        297.29254150390625,
        314.52410888671875,
        0.6293796896934509
      ],
      [
        290.9452819824219,
        310.3515319824219,
        0.640852153301239
      ],
      [
        304.86236572265625,
        313.2810974121094,
        0.6921122670173645
      ],
      [
        284.668212890625,
        266.52752685546875,
        0.9321146607398987
      ],
      [
        284.6539001464844,
        237.49722290039062,
        0.4642881155014038
      ],
      [
        298.547607421875,
        218.42587280273438,
        0.26315996050834656
      ],
      [
        182.9195098876953,
        312.5831604003906,
        0.4563772678375244
      ],
      [
   

## Save captured packets

This is useful if you want to inspect the stream shape after a test run.

In [7]:
out_path = Path("mock_unity_captured_packets.json")
out_path.write_text(json.dumps(packets, default=str, indent=2), encoding="utf-8")
print("Saved", len(packets), "packet(s) to", out_path.resolve())

Saved 20 packet(s) to C:\Users\Cyril A\Desktop\Code\DeepLabCut-live-GUI\dlclivegui\processors\custom\mock_unity_captured_packets.json


## Close the client connection

In [8]:
try:
    conn.close()
    print("Connection closed")
except Exception as exc:
    print("Close failed:", exc)

Connection closed


## Optional local smoke test with a standalone mock server

Use this only if you want to validate the notebook client logic without running DLCLiveGUI.

This requires `mock_socket_processor.py` to be importable, for example by placing it next to this notebook or adding its directory to `sys.path`.

In [9]:
# Optional smoke test. Leave commented unless you have mock_socket_processor.py available.
#
# import socket
# import sys
# from multiprocessing.connection import Client
#
# from mock_socket_processor import MockSocketProcessor
#
# def free_port():
#     s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
#     s.bind(("127.0.0.1", 0))
#     port = s.getsockname()[1]
#     s.close()
#     return port
#
# port = free_port()
# mock = MockSocketProcessor(bind=("127.0.0.1", port), authkey=AUTHKEY)
# test_conn = Client(mock.address, authkey=AUTHKEY)
# test_conn.send({"cmd": "ping"})
# print(test_conn.recv())
# mock.process([[1, 2, 0.9]], frame_time=123.456)
# print(decode_payload(test_conn.recv()))
# test_conn.close()
# mock.stop()